In [ ]:
# Cell 1: Install and import libraries (UPDATED)
!pip install rasterio shapely matplotlib

import os
from pathlib import Path
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds, calculate_default_transform
from rasterio.mask import mask
from shapely.geometry import box
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import pandas as pd
import geopandas as gpd
from rasterio.features import rasterize
from rasterio.transform import from_bounds

# Clip, Resize and Align All Input Rasters

This notebook takes raw source rasters (GDP, population, land cover, CISI) at their native resolutions and coordinate systems, and produces grid-aligned GeoTIFFs at 0.10° and 0.25° over the EU bounding box (-25, 32.41) to (32, 80.91).

**Resampling methods per variable:**
- **Land cover** (categorical): `nearest` to 0.10°, `mode` to 0.25°
- **GDP** (extensive, total M USD per cell): `sum`
- **Population** (extensive, headcount per cell): `sum`
- **CISI** (intensive, index 0–1): `nearest` (grid alignment only, native 0.10°)

**Outputs** (all in `READY_data/`)
- `inputs/2019_gdp_aligned_010.tif`, `_025.tif`
- `inputs/2020_pop_aligned_010.tif`, `_025.tif`
- `clipped_landuse_data/clipped_*.tif` (0.10°)
- `clipped_landuse_data_025/clipped_*_025.tif` (0.25°)
- `labels/2024_CISI_010deg_nearest.tif`, `_025deg_nearest.tif`

In [6]:
# Cell 2: Define parameters
min_lon = -25
max_lon = 32
min_lat = 32.41
max_lat = 80.91
crs = "EPSG:4326"

# Define folders
landuse_data = "RAW_data/landuse_data"
clipped_landuse_data = "READY_data/clipped_landuse_data"
clipped_landuse_data_025 = "READY_data/clipped_landuse_data_025"

# Create output folders
os.makedirs(clipped_landuse_data, exist_ok=True)
os.makedirs(clipped_landuse_data_025, exist_ok=True)

print(f"Input folder: {landuse_data}")
print(f"Clipped 0.1° folder: {clipped_landuse_data}")
print(f"Clipped 0.25° folder: {clipped_landuse_data_025}")
print(f"Clipping bounds: ({min_lon}, {min_lat}) to ({max_lon}, {max_lat})")
print(f"Output CRS: {crs}")

Input folder: RAW_data/landuse_data
Clipped 0.1° folder: READY_data/clipped_landuse_data
Clipped 0.25° folder: READY_data/clipped_landuse_data_025
Clipping bounds: (-25, 32.41) to (32, 80.91)
Output CRS: EPSG:4326


In [7]:
# Cell 3: FIXED - Clip and reproject while maintaining proper resolution
geotiff_files = list(Path(landuse_data).glob("*.tif")) + list(Path(landuse_data).glob("*.tiff"))

print(f"Found {len(geotiff_files)} GeoTIFF files\n")

# Check the CRS of the first file
with rasterio.open(geotiff_files[0]) as src:
    print(f"Source file CRS: {src.crs}")
    print(f"Target CRS: {crs}")
    print(f"Target bounds: ({min_lon}, {min_lat}) to ({max_lon}, {max_lat})\n")

print("=" * 70)
print("CLIPPING FILES TO BOUNDING BOX")
print("=" * 70)

for i, geotiff_file in enumerate(geotiff_files, 1):
    print(f"\n[{i}/{len(geotiff_files)}] Processing: {geotiff_file.name}")
    
    clipped_file = os.path.join(clipped_landuse_data, f"clipped_{geotiff_file.name}")
    
    try:
        with rasterio.open(geotiff_file) as src:
            # Check if reprojection is needed
            if src.crs != crs:
                print(f"  Reprojecting from {src.crs} to {crs}...")
                
                # Calculate output dimensions to maintain approximately 0.1° resolution
                # Use 0.1° as target resolution
                target_res = 0.1
                dst_width = int((max_lon - min_lon) / target_res)
                dst_height = int((max_lat - min_lat) / target_res)
                
                # Calculate transform for exact bounds and dimensions
                dst_transform = rasterio.transform.from_bounds(
                    min_lon, min_lat, max_lon, max_lat,
                    dst_width, dst_height
                )
                
                # Create output array
                reprojected_data = np.empty((src.count, dst_height, dst_width), dtype=src.dtypes[0])
                
                # Reproject with specific dimensions
                reproject(
                    source=rasterio.band(src, 1),
                    destination=reprojected_data,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=dst_transform,
                    dst_crs=crs,
                    resampling=Resampling.nearest
                )
                
                # Update metadata
                out_meta = src.meta.copy()
                out_meta.update({
                    "height": dst_height,
                    "width": dst_width,
                    "transform": dst_transform,
                    "crs": crs,
                    "compress": "lzw"
                })
                
                # Write output
                with rasterio.open(clipped_file, "w", **out_meta) as dst:
                    dst.write(reprojected_data)
                
                print(f"  ✓ Clipped and reprojected!")
                print(f"    Output: {dst_width} x {dst_height} pixels at ~0.1° resolution")
            else:
                # Same CRS, just clip
                bbox = box(min_lon, min_lat, max_lon, max_lat)
                clipped_data, clipped_transform = mask(src, [bbox], crop=True)
                
                out_meta = src.meta.copy()
                out_meta.update({
                    "height": clipped_data.shape[1],
                    "width": clipped_data.shape[2],
                    "transform": clipped_transform,
                    "crs": crs,
                    "compress": "lzw"
                })
                
                with rasterio.open(clipped_file, "w", **out_meta) as dst:
                    dst.write(clipped_data)
                
                print(f"  ✓ Clipped!")
                print(f"    Output: {clipped_data.shape[2]} x {clipped_data.shape[1]} pixels")
            
    except Exception as e:
        print(f"  ✗ Error: {str(e)}")

print("\n" + "=" * 70)
print(f"✓ Clipping complete! Files saved to: {clipped_landuse_data}")

Found 16 GeoTIFF files

Source file CRS: ESRI:54004
Target CRS: EPSG:4326
Target bounds: (-25, 32.41) to (32, 80.91)

CLIPPING FILES TO BOUNDING BOX

[1/16] Processing: history_2020.tif
  Reprojecting from ESRI:54004 to EPSG:4326...
  ✓ Clipped and reprojected!
    Output: 570 x 485 pixels at ~0.1° resolution

[2/16] Processing: ssp1_26_2030.tif
  Reprojecting from ESRI:54004 to EPSG:4326...
  ✓ Clipped and reprojected!
    Output: 570 x 485 pixels at ~0.1° resolution

[3/16] Processing: ssp1_26_2050.tif
  Reprojecting from ESRI:54004 to EPSG:4326...
  ✓ Clipped and reprojected!
    Output: 570 x 485 pixels at ~0.1° resolution

[4/16] Processing: ssp1_26_2100.tif
  Reprojecting from ESRI:54004 to EPSG:4326...
  ✓ Clipped and reprojected!
    Output: 570 x 485 pixels at ~0.1° resolution

[5/16] Processing: ssp2_45_2030.tif
  Reprojecting from ESRI:54004 to EPSG:4326...
  ✓ Clipped and reprojected!
    Output: 570 x 485 pixels at ~0.1° resolution

[6/16] Processing: ssp2_45_2050.tif
  Re

# Transforming clipped_landuse_data to 0.25 degrees

In [8]:
clipped_files = list(Path(clipped_landuse_data).glob("*.tif")) + list(Path(clipped_landuse_data).glob("*.tiff"))

print(f"\nFound {len(clipped_files)} clipped files\n")
print("=" * 70)
print("RESAMPLING TO 0.25° RESOLUTION")
print("=" * 70)

for i, clipped_file in enumerate(clipped_files, 1):
    print(f"\n[{i}/{len(clipped_files)}] Resampling: {clipped_file.name}")
    
    # Create output filename with _025 suffix
    output_name = clipped_file.stem + "_025.tif"
    resampled_file = os.path.join(clipped_landuse_data_025, output_name)
    
    try:
        with rasterio.open(clipped_file) as src:
            # Calculate new dimensions for 0.25 degree resolution
            new_width = int((max_lon - min_lon) / 0.25)
            new_height = int((max_lat - min_lat) / 0.25)
            
            # Calculate new transform
            new_transform = rasterio.transform.from_bounds(
                min_lon, min_lat, max_lon, max_lat,
                new_width, new_height
            )
            
            # Create output array
            resampled_data = np.empty(
                (src.count, new_height, new_width),
                dtype=src.dtypes[0]
            )
            
            # Resample using MODE (most common value for categorical land use data)
            reproject(
                source=src.read(),
                destination=resampled_data,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=new_transform,
                dst_crs=crs,
                resampling=Resampling.mode
            )
            
            # Write output
            with rasterio.open(
                resampled_file,
                'w',
                driver='GTiff',
                height=new_height,
                width=new_width,
                count=src.count,
                dtype=src.dtypes[0],
                crs=crs,
                transform=new_transform,
                compress='lzw'
            ) as dst:
                dst.write(resampled_data)
            
            print(f"  ✓ Resampled!")
            print(f"    Input (0.1°): {src.width} x {src.height} pixels")
            print(f"    Output (0.25°): {new_width} x {new_height} pixels")
            
    except Exception as e:
        print(f"  ✗ Error: {str(e)}")

print("\n" + "=" * 70)
print(f"✓ Resampling complete! Files saved to: {clipped_landuse_data_025}")


Found 16 clipped files

RESAMPLING TO 0.25° RESOLUTION

[1/16] Resampling: clipped_history_2020.tif
  ✓ Resampled!
    Input (0.1°): 570 x 485 pixels
    Output (0.25°): 228 x 194 pixels

[2/16] Resampling: clipped_ssp1_26_2030.tif
  ✓ Resampled!
    Input (0.1°): 570 x 485 pixels
    Output (0.25°): 228 x 194 pixels

[3/16] Resampling: clipped_ssp1_26_2050.tif
  ✓ Resampled!
    Input (0.1°): 570 x 485 pixels
    Output (0.25°): 228 x 194 pixels

[4/16] Resampling: clipped_ssp1_26_2100.tif
  ✓ Resampled!
    Input (0.1°): 570 x 485 pixels
    Output (0.25°): 228 x 194 pixels

[5/16] Resampling: clipped_ssp2_45_2030.tif
  ✓ Resampled!
    Input (0.1°): 570 x 485 pixels
    Output (0.25°): 228 x 194 pixels

[6/16] Resampling: clipped_ssp2_45_2050.tif
  ✓ Resampled!
    Input (0.1°): 570 x 485 pixels
    Output (0.25°): 228 x 194 pixels

[7/16] Resampling: clipped_ssp2_45_2100.tif
  ✓ Resampled!
    Input (0.1°): 570 x 485 pixels
    Output (0.25°): 228 x 194 pixels

[8/16] Resampling: 

# Clip, resize GDP to CISI reference grid

In [ ]:
with rasterio.open(r"READY_data\inputs\2019_gdp_original_chen_gao.tif") as src:
    print("Native resolution:", src.res)
    print("CRS:", src.crs)

with rasterio.open("READY_data/inputs/2019_gdp_aligned_010.tif") as dst:
    print("Target resolution:", dst.res)

In [ ]:
# Realign GDP to CISI reference grid (0.10° and 0.25°)
# GDP is total per cell (extensive) — use Resampling.sum
original_gdp_file = r"READY_data\inputs\2019_gdp_original_chen_gao.tif"
output_gdp_010 = "READY_data/inputs/2019_gdp_aligned_010.tif"
output_gdp_025 = "READY_data/inputs/2019_gdp_aligned_025.tif"

print("Realigning GDP to CISI reference grid\n")
print("=" * 70)

# Create 0.10 degree version
print("Creating 0.1° version...")
target_width = int((max_lon - min_lon) / 0.1)
target_height = int((max_lat - min_lat) / 0.1)

with rasterio.open(original_gdp_file) as src:
    dst_transform = rasterio.transform.from_bounds(
        min_lon, min_lat, max_lon, max_lat,
        target_width, target_height
    )
    
    aligned_data = np.empty((1, target_height, target_width), dtype=src.dtypes[0])
    
    reproject(
        source=rasterio.band(src, 1),
        destination=aligned_data[0],
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs=crs,
        resampling=Resampling.sum
    )
    
    with rasterio.open(
        output_gdp_010,
        'w',
        driver='GTiff',
        height=target_height,
        width=target_width,
        count=1,
        dtype=src.dtypes[0],
        crs=crs,
        transform=dst_transform,
        compress='lzw'
    ) as dst:
        dst.write(aligned_data)
    
    print(f"Created 0.10 degree version: {target_height} x {target_width}")
    print(f"  Bounds: ({min_lon}, {min_lat}) to ({max_lon}, {max_lat})")

# Create 0.25° version
print("\nCreating 0.25° version...")
target_width_025 = int((max_lon - min_lon) / 0.25)
target_height_025 = int((max_lat - min_lat) / 0.25)

with rasterio.open(original_gdp_file) as src:
    dst_transform_025 = rasterio.transform.from_bounds(
        min_lon, min_lat, max_lon, max_lat,
        target_width_025, target_height_025
    )
    
    aligned_data_025 = np.empty((1, target_height_025, target_width_025), dtype=src.dtypes[0])
    
    reproject(
        source=rasterio.band(src, 1),
        destination=aligned_data_025[0],
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=dst_transform_025,
        dst_crs=crs,
        resampling=Resampling.sum
    )
    
    with rasterio.open(
        output_gdp_025,
        'w',
        driver='GTiff',
        height=target_height_025,
        width=target_width_025,
        count=1,
        dtype=src.dtypes[0],
        crs=crs,
        transform=dst_transform_025,
        compress='lzw'
    ) as dst:
        dst.write(aligned_data_025)
    
    print(f"Created 0.25° version: {target_height_025} x {target_width_025}")
    print(f"  Bounds: ({min_lon}, {min_lat}) to ({max_lon}, {max_lat})")

print("\n" + "=" * 70)
print("GDP files realigned successfully!")

# Clip, resize POP to CISI reference grid

In [ ]:
# Check original population file
original_pop_file = r"RAW_data\Pop_ppp_2020_1km_Aggregated.tif"

print("Checking original population file\n")
print("=" * 70)

with rasterio.open(original_pop_file) as src:
    print(f"File: {Path(original_pop_file).name}")
    print(f"  Shape: {src.height} x {src.width}")
    print(f"  Bounds: ({src.bounds.left:.2f}, {src.bounds.bottom:.2f}) to ({src.bounds.right:.2f}, {src.bounds.top:.2f})")
    print(f"  Resolution: {src.res}")
    print(f"  CRS: {src.crs}")
    print()

print("=" * 70)
print("\nTarget alignment (CISI reference grid):")
print(f"  Bounds: ({min_lon}, {min_lat}) to ({max_lon}, {max_lat})")
print(f"  Resolution 0.1°: 485 x 570 pixels")
print(f"  Resolution 0.25°: 194 x 228 pixels")
print(f"  CRS: {crs}")

In [ ]:
# Realign population to CISI reference grid (0.10° and 0.25°)
# Population is headcount per cell (extensive) — use Resampling.sum
original_pop_file = r"RAW_data\Pop_ppp_2020_1km_Aggregated.tif"
output_pop_010 = "READY_data/inputs/2020_pop_aligned_010.tif"
output_pop_025 = "READY_data/inputs/2020_pop_aligned_025.tif"

print("Realigning population to CISI reference grid\n")
print("=" * 70)

# Create 0.1° version
print("Creating 0.1° version...")
target_width = int((max_lon - min_lon) / 0.1)
target_height = int((max_lat - min_lat) / 0.1)

with rasterio.open(original_pop_file) as src:
    dst_transform = rasterio.transform.from_bounds(
        min_lon, min_lat, max_lon, max_lat,
        target_width, target_height
    )
    
    aligned_data = np.empty((1, target_height, target_width), dtype=src.dtypes[0])
    
    reproject(
        source=rasterio.band(src, 1),
        destination=aligned_data[0],
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=dst_transform,
        dst_crs=crs,
        resampling=Resampling.sum
    )
    
    with rasterio.open(
        output_pop_010,
        'w',
        driver='GTiff',
        height=target_height,
        width=target_width,
        count=1,
        dtype=src.dtypes[0],
        crs=crs,
        transform=dst_transform,
        compress='lzw'
    ) as dst:
        dst.write(aligned_data)
    
    print(f"Created 0.1° version: {target_height} x {target_width}")
    print(f"  Bounds: ({min_lon}, {min_lat}) to ({max_lon}, {max_lat})")

# Create 0.25° version
print("\nCreating 0.25° version...")
target_width_025 = int((max_lon - min_lon) / 0.25)
target_height_025 = int((max_lat - min_lat) / 0.25)

with rasterio.open(original_pop_file) as src:
    dst_transform_025 = rasterio.transform.from_bounds(
        min_lon, min_lat, max_lon, max_lat,
        target_width_025, target_height_025
    )
    
    aligned_data_025 = np.empty((1, target_height_025, target_width_025), dtype=src.dtypes[0])
    
    reproject(
        source=rasterio.band(src, 1),
        destination=aligned_data_025[0],
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=dst_transform_025,
        dst_crs=crs,
        resampling=Resampling.sum
    )
    
    with rasterio.open(
        output_pop_025,
        'w',
        driver='GTiff',
        height=target_height_025,
        width=target_width_025,
        count=1,
        dtype=src.dtypes[0],
        crs=crs,
        transform=dst_transform_025,
        compress='lzw'
    ) as dst:
        dst.write(aligned_data_025)
    
    print(f"Created 0.25° version: {target_height_025} x {target_width_025}")
    print(f"  Bounds: ({min_lon}, {min_lat}) to ({max_lon}, {max_lat})")

print("\n" + "=" * 70)
print("Population files realigned successfully!")

# Align CISI labels to reference grid

In [ ]:
# 0.1°
with rasterio.open(r'READY_data\inputs\2019_gdp_aligned_010.tif') as ref:
    profile = ref.profile.copy()
    profile.update({'dtype': 'float32', 'nodata': np.nan})
    
    with rasterio.open(r'CISI_zenodo\CISI\010_degree\europe.tif') as src:
        out = np.empty(ref.shape, dtype=np.float32)
        out.fill(np.nan)
        
        reproject(
            rasterio.band(src, 1), out,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            resampling=Resampling.nearest
        )
        
        print(f"0.1° Max: {np.nanmax(out):.4f}")
        
        with rasterio.open(r'READY_data\labels\CISI_010deg.tif', 'w', **profile) as dst:
            dst.write(out, 1)

# 0.25°
with rasterio.open(r'READY_data\inputs\2019_gdp_aligned_025.tif') as ref:
    profile = ref.profile.copy()
    profile.update({'dtype': 'float32', 'nodata': np.nan})
    
    with rasterio.open(r'CISI_zenodo\CISI\025_degree\europe.tif') as src:
        out = np.empty(ref.shape, dtype=np.float32)
        out.fill(np.nan)
        
        reproject(
            rasterio.band(src, 1), out,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            resampling=Resampling.nearest
        )
        
        print(f"0.25° Max: {np.nanmax(out):.4f}")
        
        with rasterio.open(r'READY_data\labels\CISI_025deg.tif', 'w', **profile) as dst:
            dst.write(out, 1)

print("Done")

In [ ]:
files_010 = {
    'GDP': r'READY_data\inputs\2019_gdp_aligned_010.tif',
    'Pop': r'READY_data\inputs\2020_pop_aligned_010.tif',
    'LC': r'READY_data\clipped_landuse_data\clipped_history_2020.tif',
    'CISI': r'READY_data\labels\CISI_010deg.tif'
}

files_025 = {
    'GDP': r'READY_data\inputs\2019_gdp_aligned_025.tif',
    'Pop': r'READY_data\inputs\2020_pop_aligned_025.tif',
    'LC': r'READY_data\clipped_landuse_data_025\clipped_history_2020_025.tif',
    'CISI': r'READY_data\labels\CISI_025deg.tif'
}

print("0.1° FILES:")
shapes = {}
bounds = {}
crs = {}

for name, path in files_010.items():
    with rasterio.open(path) as src:
        shapes[name] = src.shape
        bounds[name] = src.bounds
        crs[name] = src.crs
        print(f"{name:5s}: {src.shape}, {src.bounds}")

all_match = len(set(shapes.values())) == 1 and len(set(bounds.values())) == 1 and len(set(crs.values())) == 1
print(f"All aligned: {all_match}\n")

print("0.25° FILES:")
shapes = {}
bounds = {}
crs = {}

for name, path in files_025.items():
    with rasterio.open(path) as src:
        shapes[name] = src.shape
        bounds[name] = src.bounds
        crs[name] = src.crs
        print(f"{name:5s}: {src.shape}, {src.bounds}")

all_match = len(set(shapes.values())) == 1 and len(set(bounds.values())) == 1 and len(set(crs.values())) == 1
print(f"All aligned: {all_match}")

In [ ]:
feather_path = r'CISI_zenodo\CISI\010_degree\CISI_europe.feather'
df = pd.read_feather(feather_path)

# Get CISI stats
cisi = df['CISI'].values
print(f"CISI Max: {cisi.max():.4f}")
print(f"CISI at 99.9%: {np.percentile(cisi, 99.9):.4f}")
print(f"Pixels with CISI = 1.0: {(cisi == 1.0).sum()}")
print(f"Pixels with CISI > 0.8: {(cisi > 0.8).sum()}")

# Extract coordinates from geometry
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkb(df['geometry']))

bounds = gdf.total_bounds  # [minx, miny, maxx, maxy]
print(f"\nFeather bounds:")
print(f"  West: {bounds[0]:.2f}")
print(f"  South: {bounds[1]:.2f}")
print(f"  East: {bounds[2]:.2f}")
print(f"  North: {bounds[3]:.2f}")

# Your study area
with rasterio.open(r'READY_data\inputs\2019_gdp_aligned_010.tif') as src:
    your_bounds = src.bounds
    print(f"\nYour study area:")
    print(f"  West: {your_bounds.left:.2f}")
    print(f"  South: {your_bounds.bottom:.2f}")
    print(f"  East: {your_bounds.right:.2f}")
    print(f"  North: {your_bounds.top:.2f}")

# Check where max CISI pixels are
high_cisi = gdf[gdf['CISI'] > 0.8]
high_bounds = high_cisi.total_bounds
print(f"\nHigh CISI (>0.8) pixels bounds:")
print(f"  West: {high_bounds[0]:.2f}")
print(f"  South: {high_bounds[1]:.2f}")
print(f"  East: {high_bounds[2]:.2f}")
print(f"  North: {high_bounds[3]:.2f}")

In [ ]:
feather_path = r'CISI_zenodo\CISI\010_degree\CISI_europe.feather'
df = pd.read_feather(feather_path)
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkb(df['geometry']))

# Your study area bounds
west, south, east, north = -25.0, 32.41, 32.0, 80.91

# Filter to high CISI in your area
high_cisi = gdf[gdf['CISI'] > 0.8]
centroids = high_cisi.geometry.centroid

for idx, row in high_cisi.iterrows():
    cent = centroids.iloc[high_cisi.index.get_loc(idx)]
    lon, lat = cent.x, cent.y
    in_bounds = (west <= lon <= east) and (south <= lat <= north)
    print(f"CISI: {row['CISI']:.4f}, Lon: {lon:.2f}, Lat: {lat:.2f}, In bounds: {in_bounds}")

# Check which CISI file has best boundaries

In [ ]:
target = (-25.0, 32.41, 32.0, 80.91)

# TIF files
tifs = {
    'Zenodo 0.1°': r'CISI_zenodo\CISI\010_degree\europe.tif',
    'Zenodo 0.25°': r'CISI_zenodo\CISI\025_degree\europe.tif',
    'Your 0.1°': r'READY_data\labels\2024_CISI_010deg.tif',
    'Your 0.25°': r'READY_data\labels\2024_CISI_025deg.tif',
}

for name, path in tifs.items():
    try:
        with rasterio.open(path) as src:
            b = src.bounds
            print(f"{name}: ({b.left:.1f}, {b.bottom:.1f}, {b.right:.1f}, {b.top:.1f})")
    except:
        print(f"{name}: NOT FOUND")

# Feather files
feathers = {
    'Feather 0.1°': r'CISI_zenodo\CISI\010_degree\CISI_europe.feather',
    'Feather 0.25°': r'CISI_zenodo\CISI\025_degree\CISI_europe.feather',
}

for name, path in feathers.items():
    try:
        df = pd.read_feather(path)
        gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkb(df['geometry']))
        b = gdf.total_bounds
        print(f"{name}: ({b[0]:.1f}, {b[1]:.1f}, {b[2]:.1f}, {b[3]:.1f})")
    except:
        print(f"{name}: NOT FOUND")

print(f"\nTarget: {target}")

In [ ]:
with rasterio.open(r'READY_data\inputs\2019_gdp_aligned_025.tif') as ref:
    profile = ref.profile.copy()
    profile.update({'dtype': 'float32', 'nodata': np.nan})
    
    with rasterio.open(r'CISI_zenodo\CISI\025_degree\europe.tif') as src:
        out = np.empty(ref.shape, dtype=np.float32)
        out.fill(np.nan)
        
        reproject(
            rasterio.band(src, 1), out,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            resampling=Resampling.bilinear
        )
        
        with rasterio.open(r'READY_data\labels\CISI_025deg_bilinear.tif', 'w', **profile) as dst:
            dst.write(out, 1)
        
        print(f"Max: {np.nanmax(out):.4f}")

print("Done. Use: CISI_025deg_bilinear.tif")

# Nearest Neighbor

In [ ]:
# 0.25° with nearest neighbor
with rasterio.open(r'READY_data\inputs\2019_gdp_aligned_025.tif') as ref:
    profile = ref.profile.copy()
    profile.update({'dtype': 'float32', 'nodata': np.nan})
    
    with rasterio.open(r'CISI_zenodo\CISI\025_degree\europe.tif') as src:
        out = np.empty(ref.shape, dtype=np.float32)
        out.fill(np.nan)
        
        reproject(
            rasterio.band(src, 1), out,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            resampling=Resampling.nearest
        )
        
        with rasterio.open(r'READY_data\labels\2024_CISI_025deg_nearest.tif', 'w', **profile) as dst:
            dst.write(out, 1)
        
        print(f"0.25° Nearest - Max: {np.nanmax(out):.4f}")

# 0.1° with nearest neighbor
with rasterio.open(r'READY_data\inputs\2019_gdp_aligned_010.tif') as ref:
    profile = ref.profile.copy()
    profile.update({'dtype': 'float32', 'nodata': np.nan})
    
    with rasterio.open(r'CISI_zenodo\CISI\010_degree\europe.tif') as src:
        out = np.empty(ref.shape, dtype=np.float32)
        out.fill(np.nan)
        
        reproject(
            rasterio.band(src, 1), out,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=ref.transform, dst_crs=ref.crs,
            resampling=Resampling.nearest
        )
        
        with rasterio.open(r'READY_data\labels\2024_CISI_010deg_nearest.tif', 'w', **profile) as dst:
            dst.write(out, 1)
        
        print(f"0.1° Nearest - Max: {np.nanmax(out):.4f}")

print("Done. Use: 2024_CISI_025deg_nearest.tif and 2024_CISI_010deg_nearest.tif")

# Transform .feather to .tif

In [ ]:
# Load feather file
df = pd.read_feather(r'READY_data\labels\CISI_exposure_Europe_025deg.feather')

print("Feather file contents:")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nCISI stats:")
print(f"  Min: {df['CISI'].min():.4f}")
print(f"  Max: {df['CISI'].max():.4f}")
print(f"  Mean: {df['CISI'].mean():.4f}")
print(f"  Median: {df['CISI'].median():.4f}")
print(f"  99th percentile: {df['CISI'].quantile(0.99):.4f}")

print(f"\nFirst 5 rows:")
print(df[['CISI']].head(20))

# Compare to your TIF file
with rasterio.open(r'READY_data\labels\CISI_025deg_bilinear.tif') as src:
    tif_data = src.read(1)
    print(f"\nCurrent TIF file stats:")
    print(f"  Max: {np.nanmax(tif_data):.4f}")
    print(f"  Mean: {np.nanmean(tif_data):.4f}")

In [ ]:
# Load feather
df = pd.read_feather(r'READY_data\labels\CISI_exposure_Europe_025deg.feather')
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkb(df['geometry']))
gdf = gdf.set_crs('EPSG:4326')

# Load reference grid (to match GDP/Pop alignment)
with rasterio.open(r'READY_data\inputs\2019_gdp_aligned_025.tif') as ref:
    profile = ref.profile.copy()
    profile.update({'dtype': 'float32', 'nodata': np.nan})
    
    # Rasterize CISI from feather
    shapes = [(geom, value) for geom, value in zip(gdf.geometry, gdf['CISI'])]
    
    cisi_array = rasterize(
        shapes,
        out_shape=ref.shape,
        transform=ref.transform,
        fill=np.nan,
        dtype=np.float32
    )
    
    # Save corrected CISI
    with rasterio.open(r'READY_data\labels\CISI_025deg_corrected.tif', 'w', **profile) as dst:
        dst.write(cisi_array, 1)
    
    print(f"Corrected CISI saved!")
    print(f"Max: {np.nanmax(cisi_array):.4f}")
    print(f"Mean: {np.nanmean(cisi_array):.4f}")